In [2]:
import pandas as pd
import optuna
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import shap
import wandb
from Optune_simulation_env import get_best_params, walk_forward_predict_test
from utils import load_data
from scipy.stats import ttest_rel

c:\Users\local_user\anaconda3\envs\EnergyPrices\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
N_TRIALS = 30
FINAL_TEST_DAYS = 30
OPTUNA_VAL_DAYS = 30
N_Optuna_Runs = 17
COUNTRY = "HU"

In [4]:
real_ds = load_data("real", COUNTRY)

In [5]:
display(real_ds)
print(real_ds.columns)

,last_y,lag_1_t0,lag_4_t0,lag_8_t0,lag_24_t0,lag_96_t0,lag_192_t0,lag_672_t0,ramp_1h_t0,ramp_6h_t0,...,daily_weight_lag_1w,hour_weight_lag_1d,hour_weight_lag_2d,hour_weight_lag_1w,daily_avg_weight_lag_1d,daily_avg_weight_lag_2d,daily_avg_weight_lag_1w,hour_avg_weight_lag_1d,hour_avg_weight_lag_2d,hour_avg_weight_lag_1w
ts,,,,,,,,,,,,,,,,,,,,,
2025-10-23 00:00:00+02:00,105.92,114.01,112.68,107.79,273.76,105.62,131.97,116.57,-6.76,-167.84,...,0.009603,0.274744,0.274857,0.265149,0.767645,0.866840,0.921894,1.098978,1.099429,1.060595
2025-10-23 00:15:00+02:00,105.92,114.01,112.68,107.79,273.76,100.00,122.11,117.07,-6.76,-167.84,...,0.009644,0.260125,0.254322,0.266286,0.726799,0.802075,0.925849,1.040502,1.017287,1.065144
2025-10-23 00:30:00+02:00,105.92,114.01,112.68,107.79,273.76,89.81,120.99,106.52,-6.76,-167.84,...,0.008775,0.233619,0.251989,0.242289,0.652738,0.794718,0.842414,0.934474,1.007956,0.969157
2025-10-23 00:45:00+02:00,105.92,114.01,112.68,107.79,273.76,89.00,105.07,99.48,-6.76,-167.84,...,0.008195,0.231512,0.218832,0.226276,0.646851,0.690148,0.786738,0.926046,0.875328,0.905104
2025-10-23 01:00:00+02:00,105.92,114.01,112.68,107.79,273.76,100.75,101.19,108.09,-6.76,-167.84,...,0.008904,0.263213,0.270576,0.260671,0.732250,0.664663,0.854830,1.052852,1.082304,1.042686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-13 22:45:00+01:00,110.99,126.34,120.48,128.76,299.49,120.48,131.51,121.74,-9.49,-188.50,...,0.011154,0.204759,0.236895,0.232942,0.894156,1.184665,1.070824,0.819035,0.947581,0.931767
2026-03-13 23:00:00+01:00,110.99,126.34,120.48,128.76,299.49,149.77,138.85,140.81,-9.49,-188.50,...,0.012902,0.297216,0.295696,0.270113,1.111535,1.250785,1.238564,1.188863,1.182784,1.080453
2026-03-13 23:15:00+01:00,110.99,126.34,120.48,128.76,299.49,116.81,114.68,131.12,-9.49,-188.50,...,0.012014,0.231807,0.244223,0.251525,0.866919,1.033057,1.153331,0.927229,0.976894,1.006100


Index(['last_y', 'lag_1_t0', 'lag_4_t0', 'lag_8_t0', 'lag_24_t0', 'lag_96_t0',
       'lag_192_t0', 'lag_672_t0', 'ramp_1h_t0', 'ramp_6h_t0', 'ramp_1d_t0',
       'roll_mean_24_t0', 'roll_std_24_t0', 'roll_mean_96_t0',
       'roll_std_96_t0', 'roll_mean_672_t0', 'roll_std_672_t0', 'h',
       'q_in_hour_target', 'qod_target', 'hod_target', 'dow_target',
       'month_target', 'is_weekend_target', 'q_in_hour_sin', 'q_in_hour_cos',
       'qod_sin', 'qod_cos', 'hod_sin', 'hod_cos', 'dow_sin', 'dow_cos',
       'month_sin', 'month_cos', 'load_fc_target', 'renewables_wind_fc',
       'renewables_solar_fc', 'load_ramp_1h_target', 'load_ramp_6h_target',
       'load_ramp_12h_target', 'load_day_mean', 'load_day_max', 'load_day_min',
       'y_target', 'is_synthetic', 'is_observed', 'day', 'daily_weight_lag_1d',
       'daily_weight_lag_2d', 'daily_weight_lag_1w', 'hour_weight_lag_1d',
       'hour_weight_lag_2d', 'hour_weight_lag_1w', 'daily_avg_weight_lag_1d',
       'daily_avg_weight_lag_2

In [6]:
test_df = real_ds[["y_target", "lag_96_t0", "day"]]
print(test_df)

                           y_target  lag_96_t0                        day
ts                                                                       
2025-10-23 00:00:00+02:00    117.43     105.62  2025-10-23 00:00:00+02:00
2025-10-23 00:15:00+02:00    111.14     100.00  2025-10-23 00:00:00+02:00
2025-10-23 00:30:00+02:00    103.57      89.81  2025-10-23 00:00:00+02:00
2025-10-23 00:45:00+02:00    100.16      89.00  2025-10-23 00:00:00+02:00
2025-10-23 01:00:00+02:00    114.63     100.75  2025-10-23 00:00:00+02:00
...                             ...        ...                        ...
2026-03-13 22:45:00+01:00    116.90     120.48  2026-03-13 00:00:00+01:00
2026-03-13 23:00:00+01:00    133.69     149.77  2026-03-13 00:00:00+01:00
2026-03-13 23:15:00+01:00    115.61     116.81  2026-03-13 00:00:00+01:00
2026-03-13 23:30:00+01:00    115.10     126.34  2026-03-13 00:00:00+01:00
2026-03-13 23:45:00+01:00    103.35     110.99  2026-03-13 00:00:00+01:00

[13632 rows x 3 columns]


In [ ]:
all_days = np.array(sorted(test_df["day"].unique()))
final_test_days = all_days[-FINAL_TEST_DAYS:]
tune_days = all_days[:-FINAL_TEST_DAYS]

print(final_test_days)

['2026-02-12 00:00:00+01:00' '2026-02-13 00:00:00+01:00'
 '2026-02-14 00:00:00+01:00' '2026-02-15 00:00:00+01:00'
 '2026-02-16 00:00:00+01:00' '2026-02-17 00:00:00+01:00'
 '2026-02-18 00:00:00+01:00' '2026-02-19 00:00:00+01:00'
 '2026-02-20 00:00:00+01:00' '2026-02-21 00:00:00+01:00'
 '2026-02-22 00:00:00+01:00' '2026-02-23 00:00:00+01:00'
 '2026-02-24 00:00:00+01:00' '2026-02-25 00:00:00+01:00'
 '2026-02-26 00:00:00+01:00' '2026-02-27 00:00:00+01:00'
 '2026-02-28 00:00:00+01:00' '2026-03-01 00:00:00+01:00'
 '2026-03-02 00:00:00+01:00' '2026-03-03 00:00:00+01:00'
 '2026-03-04 00:00:00+01:00' '2026-03-05 00:00:00+01:00'
 '2026-03-06 00:00:00+01:00' '2026-03-07 00:00:00+01:00'
 '2026-03-08 00:00:00+01:00' '2026-03-09 00:00:00+01:00'
 '2026-03-10 00:00:00+01:00' '2026-03-11 00:00:00+01:00'
 '2026-03-12 00:00:00+01:00' '2026-03-13 00:00:00+01:00']


In [10]:
ds_test_pool  = test_df[test_df["day"].isin(final_test_days)].copy()

In [11]:
display(ds_test_pool)

,y_target,lag_96_t0,day
ts,,,
2026-02-12 00:00:00+01:00,96.36,102.95,2026-02-12 00:00:00+01:00
2026-02-12 00:15:00+01:00,94.85,104.25,2026-02-12 00:00:00+01:00
2026-02-12 00:30:00+01:00,96.79,102.89,2026-02-12 00:00:00+01:00
2026-02-12 00:45:00+01:00,94.61,102.55,2026-02-12 00:00:00+01:00
2026-02-12 01:00:00+01:00,96.36,102.88,2026-02-12 00:00:00+01:00
...,...,...,...
2026-03-13 22:45:00+01:00,116.90,120.48,2026-03-13 00:00:00+01:00
2026-03-13 23:00:00+01:00,133.69,149.77,2026-03-13 00:00:00+01:00
2026-03-13 23:15:00+01:00,115.61,116.81,2026-03-13 00:00:00+01:00


In [12]:
mae = mean_absolute_error(ds_test_pool["y_target"], ds_test_pool["lag_96_t0"])
rmse = root_mean_squared_error(y_true=ds_test_pool["y_target"], y_pred=ds_test_pool["lag_96_t0"])

print(mae)
print(rmse)


21.602489583333334
31.164018339464825
